# 排程總管 `_Scheduler.ipynb`

所有定期工作集中在這裡管理：**第 1 區**看工作清單與規則，**第 4 區**看目前狀態。

| 執行方式 | 作用 |
|---|---|
| 直接在 Jupyter 執行全部 | 只顯示狀態，不會跑任何工作（`JOB_MODE` 預設 `status`） |
| `runScheduler.bat auto` | 檢查每個工作，該跑的才跑（Windows 排程用這個） |
| `runScheduler.bat status` | 只看狀態 |
| `runScheduler.bat run 工作名` | 強制執行指定工作，例如 `runScheduler.bat run profit` |

判斷都看「上次成功時間 / 資料新舊」，不看今天星期幾，所以電腦哪天沒開機，下次開機會自動補跑。
- 每次執行紀錄：`Result/_scheduler/logs/`（保留 30 天），出錯時打開對應的 `.ipynb` 看
- 狀態：`Result/_scheduler/state.json`

In [1]:
import os
# =============================================================
# 1. 工作清單（排程規則都在這裡；順序 = 執行順序）
# =============================================================
#   rule:
#     daily       每個交易日一次，after 時間之後才算當天（週末不算）
#     once_a_day  每天一次（不分平日假日）
#     quarterly   季財報公布期限過後，本機季別落後就跑；期限後 21 天內每 7 天再補抓一次（晚交財報）
#     remind      不自動跑，檔案超過 remind_days 天沒更新就 LINE 提醒
#   evidence: 工作的產出檔（或資料夾）。跑完要比開始時間新才算成功；沒有 state 紀錄時拿它推測上次成功時間
JOBS = {
    'main': dict(
        title='每日主流程（日K、指標、法人、選股、推薦 LINE）',
        nb=['Main.ipynb'], rule='daily', after='15:10', timeout_h=6,
        evidence='Result/DailyFiles'),
    'revenue': dict(
        title='月營收 opendata（每天抓，公布後自然更新）',
        nb=['_BaseRevenue.ipynb'], rule='once_a_day', timeout_h=1,
        evidence='Result/t187ap05_L.csv'),
    'stockinfo': dict(
        title='個股基本資料 + 合併月營收（排在 revenue 之後）',
        nb=['_CrawStockInfo.ipynb'], rule='daily', after='15:10', timeout_h=4,
        evidence='Result/StockInfo.xlsx'),
    'etf': dict(
        title='ETF 持股明細（18:00 後才有當天資料）',
        nb=['Project_ETF/ETF_Portfolio_Downloader.ipynb'], rule='daily', after='18:00', timeout_h=1,
        evidence='Project_ETF/ETF_Investment_Portfolio'),
    'profit': dict(
        title='季財報 → 拆檔 → 基本面評分',
        nb=['__stockProfitability_1_Craw.ipynb',
            '__stockProfitability_2_split.ipynb',
            '__stockProfitability_3_fundamental_score.ipynb'],
        rule='quarterly', timeout_h=6,
        evidence='Result/StockProfitability_all.xlsx'),
    'stocktype': dict(
        title='產業鏈分類（要公司 proxy，手動跑 _CrawStockType）',
        nb=['_CrawStockType.ipynb'], rule='remind', remind_days=90,
        evidence='Result/StockType.xlsx'),
    'etfbase': dict(
        title='ETF 清單 / 產業比重（手動跑 _BaseInfoETF）',
        nb=['_BaseInfoETF.ipynb'], rule='remind', remind_days=90,
        evidence='BaseInfo/ETFBaseData.csv'),
}

# 季財報法定公布期限（月, 日）→ 期限過後應該有的季別
SEASON_DEADLINES = [((3, 31), 4), ((5, 15), 1), ((8, 14), 2), ((11, 14), 3)]
CATCHUP_DAYS  = 21     # 期限後幾天內持續補抓晚交的公司
CATCHUP_EVERY = 7      # 補抓間隔（天）

SEND_LINE         = os.environ.get('SEND_LINE', '1') != '0'   # 測試時 set SEND_LINE=0
REMIND_EVERY_DAYS = 7  # 同一個提醒多久發一次
LOG_KEEP_DAYS     = 30
LOCK_STALE_HOURS  = 12 # 鎖定檔超過幾小時視為上次異常中斷

# 單次提醒：到了日期後，第一次執行 auto 時用 LINE 提醒一次（發過就不再發）
NOTES = [
    ('2026-10-31', '新排程 StockAuto 已跑一週：確認正常後，刪除桌面 runMain.bat / runCrawStockInfo.bat / runETF.bat，以及已停用的 MainStock、MainETF 排程'),
]

In [2]:
# =============================================================
# 2. 工具：路徑、狀態、鎖定、執行 notebook
# =============================================================
import os, sys, json, time, subprocess
from datetime import datetime, timedelta, date
from pathlib import Path
import pandas as pd
import psutil

BASE_DIR   = Path(r'D:\Project\Jupyter\Stock\Main')
SCHED_DIR  = BASE_DIR / 'Result' / '_scheduler'
LOG_DIR    = SCHED_DIR / 'logs'
STATE_PATH = SCHED_DIR / 'state.json'
LOCK_PATH  = SCHED_DIR / 'running.lock'
LOG_DIR.mkdir(parents=True, exist_ok=True)

# 執行模式（runScheduler.bat 傳入；直接開 notebook 預設只看狀態）
JOB_MODE = os.environ.get('JOB_MODE', 'status').strip().lower()
JOB_NAME = os.environ.get('JOB_NAME', '').strip()
NOW = datetime.now()
FMT = '%Y-%m-%d %H:%M'
print('模式:', JOB_MODE, JOB_NAME, '|', NOW.strftime(FMT))


def load_state():
    if STATE_PATH.exists():
        return json.load(open(STATE_PATH, encoding='utf-8'))
    return {}


def save_state(state):
    json.dump(state, open(STATE_PATH, 'w', encoding='utf-8'), ensure_ascii=False, indent=1)


def evidence_mtime(job):
    # 產出檔的最後修改時間；資料夾取裡面最新的檔案
    p = BASE_DIR / JOBS[job]['evidence']
    if p.is_dir():
        files = [f for f in p.iterdir() if f.is_file()]
        return datetime.fromtimestamp(max(f.stat().st_mtime for f in files)) if files else None
    return datetime.fromtimestamp(p.stat().st_mtime) if p.exists() else None


def last_success(state, job):
    # 回傳 (時間, 是否為推測)
    s = state.get(job, {}).get('last_success')
    if s:
        return datetime.strptime(s, FMT), False
    m = evidence_mtime(job)
    return m, m is not None


def acquire_lock():
    if LOCK_PATH.exists():
        lock_time = LOCK_PATH.stat().st_mtime
        age_h = (time.time() - lock_time) / 3600
        if lock_time < psutil.boot_time():
            print('鎖定檔是開機前留下的（上次執行時關機），繼續執行')
        elif age_h < LOCK_STALE_HOURS:
            print('已有排程在執行中（', LOCK_PATH.read_text(encoding='utf-8'), '），本次略過')
            return False
        else:
            print(f'鎖定檔已 {age_h:.1f} 小時，視為上次中斷，繼續執行')
    LOCK_PATH.write_text(f'pid {os.getpid()} @ {NOW.strftime(FMT)}', encoding='utf-8')
    return True


def run_notebook(job, nb_rel, timeout_h):
    # 另開程序執行，失敗不影響其他工作；執行結果存到 logs/
    nb_path = BASE_DIR / nb_rel
    out_name = f"{datetime.now().strftime('%Y%m%d_%H%M')}_{job}_{nb_path.stem}"
    cmd = [sys.executable, '-m', 'jupyter', 'nbconvert', '--to', 'notebook', '--execute', str(nb_path),
           f'--ExecutePreprocessor.timeout={int(timeout_h * 3600)}',
           '--output-dir', str(LOG_DIR), '--output', out_name]
    print(f'  ▶ {nb_rel}  →  logs/{out_name}.ipynb')
    try:
        p = subprocess.run(cmd, cwd=str(nb_path.parent), capture_output=True, timeout=timeout_h * 3600 + 600)
    except subprocess.TimeoutExpired:
        return False, f'{nb_rel} 超過 {timeout_h} 小時'
    if p.returncode != 0:
        err = p.stderr.decode('utf-8', errors='replace').strip().splitlines()
        return False, f'{nb_rel}：' + (err[-1] if err else f'exit {p.returncode}')
    return True, ''


def clean_logs():
    limit = time.time() - LOG_KEEP_DAYS * 86400
    for f in LOG_DIR.glob('*.ipynb'):
        if f.stat().st_mtime < limit:
            f.unlink()


state = load_state()

模式: auto  | 2026-10-24 16:57


In [3]:
# =============================================================
# 3. 判斷規則：現在要不要跑
# =============================================================
def last_session(hhmm, now=NOW):
    # 最近一個「已經過了 hhmm 的平日」
    h, m = map(int, hhmm.split(':'))
    t = now.replace(hour=h, minute=m, second=0, microsecond=0)
    if now < t:
        t -= timedelta(days=1)
    while t.weekday() >= 5:
        t -= timedelta(days=1)
    return t


def expected_season(today=NOW.date()):
    # 已過公布期限的最新一季 → (民國年, 季, 期限日)
    best = (today.year - 1 - 1911, 3, date(today.year - 1, 11, 14))
    for (m, d), q in SEASON_DEADLINES:
        deadline = date(today.year, m, d)
        if today > deadline:
            year = today.year - 1 if q == 4 else today.year
            best = (year - 1911, q, deadline)
    return best


def local_season():
    # StockProfitability.xlsx 只存最新一季，季別格式 '115.2Q'
    p = BASE_DIR / 'Result' / 'StockProfitability.xlsx'
    if not p.exists():
        return None
    s = pd.read_excel(p, usecols=['季別'])['季別'].dropna().astype(str).str.replace('Q', '', regex=False)
    if s.empty:
        return None
    y, q = max((int(v.split('.')[0]), int(v.split('.')[1])) for v in s)
    return (y, q)


def check_due(job, state):
    # 回傳 (要不要跑, 原因)
    cfg = JOBS[job]
    last, _ = last_success(state, job)
    rule = cfg['rule']

    if rule == 'daily':
        need = last_session(cfg['after'])
        if last is None or last < need:
            return True, f"還沒跑 {need.strftime('%m/%d')} 的資料"
        return False, f"已涵蓋 {need.strftime('%m/%d')}"

    if rule == 'once_a_day':
        if last is None or last.date() < NOW.date():
            return True, '今天還沒跑'
        return False, '今天已跑'

    if rule == 'quarterly':
        y, q, deadline = expected_season()
        have = local_season()
        if have is None or have < (y, q):
            return True, f"應有 {y}.{q}Q，本機 {'無' if have is None else f'{have[0]}.{have[1]}Q'}"
        in_window = (NOW.date() - deadline).days <= CATCHUP_DAYS
        if in_window and (last is None or (NOW - last).days >= CATCHUP_EVERY):
            return True, f'{y}.{q}Q 期限後 {CATCHUP_DAYS} 天內，補抓晚交財報'
        return False, f'已有 {y}.{q}Q'

    if rule == 'remind':
        age = None if last is None else (NOW - last).days
        if age is None or age >= cfg['remind_days']:
            return False, f"已 {'很久' if age is None else f'{age} 天'}沒更新，該手動跑了"
        return False, f'手動（{age} 天前）'
    raise ValueError(f'{job} 未知的 rule: {rule}')


def check_remind(job, state):
    # remind 類：超過天數就回傳提醒文字（REMIND_EVERY_DAYS 內只提醒一次）
    cfg = JOBS[job]
    m = evidence_mtime(job)
    age = None if m is None else (NOW - m).days
    if age is not None and age < cfg['remind_days']:
        return None
    last_remind = state.get(job, {}).get('last_remind')
    if last_remind and (NOW - datetime.strptime(last_remind, FMT)).days < REMIND_EVERY_DAYS:
        return None
    nb = ', '.join(cfg['nb'])
    return f"{job} 已 {'很久' if age is None else f'{age} 天'}沒更新 → 手動跑 {nb}"

In [4]:
# =============================================================
# 4. 狀態總覽
# =============================================================
def status_table(state):
    rows = []
    for job, cfg in JOBS.items():
        last, guessed = last_success(state, job)
        due, why = check_due(job, state)
        st = state.get(job, {})
        rows.append({
            '工作': job, '說明': cfg['title'], '規則': cfg['rule'] + (f" {cfg['after']}" if 'after' in cfg else ''),
            '上次成功': '-' if last is None else last.strftime(FMT) + ('（推測）' if guessed else ''),
            '上次結果': st.get('result', '-'), '耗時(分)': st.get('duration_min', '-'),
            '需要執行': '●' if due else '', '原因': why, '訊息': st.get('message', ''),
        })
    return pd.DataFrame(rows).set_index('工作')


pd.set_option('display.max_colwidth', 80)
for day, text in NOTES:
    print(f"{'已提醒' if text in state.get('_notes', {}) else '待提醒'} {day}：{text}")
status_table(state)

待提醒 2026-10-31：新排程 StockAuto 已跑一週：確認正常後，刪除桌面 runMain.bat / runCrawStockInfo.bat / runETF.bat，以及已停用的 MainStock、MainETF 排程


,說明,規則,上次成功,上次結果,耗時(分),需要執行,原因,訊息
工作,,,,,,,,
main,每日主流程（日K、指標、法人、選股、推薦 LINE）,daily 15:10,2026-10-24 13:20,OK,161.1,,已涵蓋 10/23,
revenue,月營收 opendata（每天抓，公布後自然更新）,once_a_day,2026-10-24 13:10,OK,0.1,,今天已跑,
stockinfo,個股基本資料 + 合併月營收（排在 revenue 之後）,daily 15:10,2026-10-24 16:01,OK,44.8,,已涵蓋 10/23,
etf,ETF 持股明細（18:00 後才有當天資料）,daily 18:00,2026-10-24 16:46,OK,0.5,,已涵蓋 10/23,
profit,季財報 → 拆檔 → 基本面評分,quarterly,2026-08-22 09:51（推測）,-,-,,已有 115.2Q,
stocktype,產業鏈分類（要公司 proxy，手動跑 _CrawStockType）,remind,2025-11-15 14:20（推測）,-,-,,已 343 天沒更新，該手動跑了,
etfbase,ETF 清單 / 產業比重（手動跑 _BaseInfoETF）,remind,2026-03-02 21:48（推測）,-,-,,已 235 天沒更新，該手動跑了,


In [5]:
# =============================================================
# 5. 執行（auto：該跑的才跑；run：強制跑 JOB_NAME）
# =============================================================
def run_job(job, state):
    cfg = JOBS[job]
    start = datetime.now()
    print(f'\n=== {job}：{cfg["title"]}（開始 {start.strftime(FMT)}）')
    st = state.setdefault(job, {})
    st['last_attempt'] = start.strftime(FMT)
    ok, msg = True, ''
    for nb_rel in cfg['nb']:              # 多支 notebook 依序跑，失敗就停
        ok, msg = run_notebook(job, nb_rel, cfg.get('timeout_h', 2))
        if not ok:
            break
    if ok:
        m = evidence_mtime(job)
        if m is None or m < start:
            ok, msg = False, f"沒有產出新檔案（{cfg['evidence']}）"
    st['duration_min'] = round((datetime.now() - start).total_seconds() / 60, 1)
    st['result'] = 'OK' if ok else '失敗'
    st['message'] = msg
    if ok:
        st['last_success'] = start.strftime(FMT)
    save_state(state)                     # 每跑完一個就存，中途關機也保留進度
    print(f"  {'✅' if ok else '❌'} {st['duration_min']} 分 {msg}")
    return job, ok, st['duration_min'], msg


results, reminders = [], []
if JOB_MODE in ('auto', 'run') and acquire_lock():
    try:
        if JOB_MODE == 'run':
            if JOB_NAME not in JOBS:
                raise ValueError(f'找不到工作 {JOB_NAME!r}，可用：{list(JOBS)}')
            results.append(run_job(JOB_NAME, state))
        else:
            for job, cfg in JOBS.items():
                if cfg['rule'] == 'remind':
                    text = check_remind(job, state)
                    if text:
                        reminders.append(text)
                        state.setdefault(job, {})['last_remind'] = NOW.strftime(FMT)
                        save_state(state)
                    continue
                due, why = check_due(job, state)   # 每次重新判斷（前面的工作可能跑很久）
                if due:
                    print(f'{job}：{why}')
                    results.append(run_job(job, state))
            sent = state.setdefault('_notes', {})
            for day, text in NOTES:
                if NOW.strftime('%Y-%m-%d') >= day and text not in sent:
                    reminders.append(text)
                    sent[text] = NOW.strftime(FMT)
                    save_state(state)
        clean_logs()
    finally:
        LOCK_PATH.unlink(missing_ok=True)
print('本次執行:', [(j, ok) for j, ok, _, _ in results] or '無', '| 提醒:', len(reminders))

本次執行: 無 | 提醒: 2


In [6]:
# =============================================================
# 6. LINE：有跑工作、有失敗或有提醒才發
# =============================================================
if (results or reminders) and SEND_LINE:
    lines = [f"🗓 排程 {NOW.strftime('%m/%d %H:%M')}（{JOB_MODE}）"]
    for job, ok, mins, msg in results:
        lines.append(f"{'✅' if ok else '❌'} {job} {mins} 分" + (f'\n   {msg}' if msg else ''))
    for text in reminders:
        lines.append(f'⏰ {text}')
    print('\n'.join(lines))
    %run _LineMsg.ipynb
    sendLineMessage('\n'.join(lines))
else:
    print('沒有需要通知的事')

🗓 排程 10/24 16:57（auto）
⏰ stocktype 已 343 天沒更新 → 手動跑 _CrawStockType.ipynb
⏰ etfbase 已 235 天沒更新 → 手動跑 _BaseInfoETF.ipynb


C:\Users\user\anaconda3\lib\site-packages\nbformat\__init__.py:92: MissingIDFieldWarning: Code cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)
C:\Users\user\AppData\Local\Temp\ipykernel_9636\2513558176.py:5: LineBotSdkDeprecatedIn30: Call to deprecated class LineBotApi. (Use v3 class; linebot.v3.<feature>. See https://github.com/line/line-bot-sdk-python/blob/master/README.rst for more details.) -- Deprecated since version 3.0.0.
  line_bot_api = LineBotApi(LINE_CHANNEL_ACCESS_TOKEN)
C:\Users\user\AppData\Local\Temp\ipykernel_9636\2513558176.py:24: LineBotSdkDeprecatedIn30: Call to deprecated method push_message. (Use 'from linebot.v3.messaging import MessagingApi' and 'MessagingApi(...).push_message(...)' instead. See https:/

消息发送成功！


In [7]:
# 執行後狀態
status_table(load_state())

,說明,規則,上次成功,上次結果,耗時(分),需要執行,原因,訊息
工作,,,,,,,,
main,每日主流程（日K、指標、法人、選股、推薦 LINE）,daily 15:10,2026-10-24 13:20,OK,161.1,,已涵蓋 10/23,
revenue,月營收 opendata（每天抓，公布後自然更新）,once_a_day,2026-10-24 13:10,OK,0.1,,今天已跑,
stockinfo,個股基本資料 + 合併月營收（排在 revenue 之後）,daily 15:10,2026-10-24 16:01,OK,44.8,,已涵蓋 10/23,
etf,ETF 持股明細（18:00 後才有當天資料）,daily 18:00,2026-10-24 16:46,OK,0.5,,已涵蓋 10/23,
profit,季財報 → 拆檔 → 基本面評分,quarterly,2026-08-22 09:51（推測）,-,-,,已有 115.2Q,
stocktype,產業鏈分類（要公司 proxy，手動跑 _CrawStockType）,remind,2025-11-15 14:20（推測）,-,-,,已 343 天沒更新，該手動跑了,
etfbase,ETF 清單 / 產業比重（手動跑 _BaseInfoETF）,remind,2026-03-02 21:48（推測）,-,-,,已 235 天沒更新，該手動跑了,
